In [1]:
RANDOM_STATE = 42
%matplotlib inline
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / "data/raw").exists() else Path.cwd().parent
assert (ROOT / "data/raw/application_record.csv").exists(), "Extraia o pacote completo antes de executar."
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from common import load, prepare, FEATURES, preprocessing
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", None)
OUT = ROOT / "data/processed"
OUT.mkdir(parents=True, exist_ok=True)
RESULTS = ROOT / "docs/resultados"
RESULTS.mkdir(parents=True, exist_ok=True)

# 02 — Pré-processamento e definição do alvo

In [2]:
a,c=load(ROOT)
d,t,conflicts=prepare(a,c)
print('IDs conflitantes removidos antes do join:',len(conflicts))
print('Interseção final:',len(d))
display(d[['TARGET','TARGET_90','MESES_HISTORICO']].describe())
display(d[FEATURES].isna().sum().to_frame('nulos'))

IDs conflitantes removidos antes do join: 47
Interseção final: 36457


,TARGET,TARGET_90,MESES_HISTORICO
count,36457.000000,36457.000000,36457.000000
mean,0.117700,0.008284,21.332392
std,0.322257,0.090638,14.911849
min,0.000000,0.000000,1.000000
25%,0.000000,0.000000,9.000000
50%,0.000000,0.000000,18.000000
75%,0.000000,0.000000,31.000000
max,1.000000,1.000000,61.000000


,nulos
FLAG_OWN_CAR,0
FLAG_OWN_REALTY,0
CNT_CHILDREN,0
NAME_INCOME_TYPE,0
NAME_EDUCATION_TYPE,0
NAME_FAMILY_STATUS,0
NAME_HOUSING_TYPE,0
FLAG_WORK_PHONE,0
FLAG_PHONE,0
FLAG_EMAIL,0


## 1. Integridade e ausências
Linhas integralmente duplicadas são removidas; IDs com cadastros divergentes são excluídos antes da junção validada 1:1. Perfis idênticos recebem o mesmo grupo sem usar o histórico. Ocupação ausente será Unknown; números ausentes serão imputados com medianas aprendidas somente nos folds de treino. 365243 é tratado como sentinela de emprego, sem afirmar que identifica exclusivamente desempregados.

## 2. Alvo
`TARGET=1` se houver STATUS em {1,2,3,4,5} em qualquer mês: atraso de 30+ dias. `TARGET_90` usa {3,4,5}: 90+ dias, incluindo 90, não estritamente superior. Zero significa ausência de atraso 30+ observado, podendo incluir atraso de 1–29 dias, C e X; não significa ausência de qualquer atraso. Clientes sem histórico não recebem rótulo zero. Históricos somente X permanecem na base, mas são uma limitação: sem crédito ativo observado, a oportunidade de atraso é distinta.

In [3]:
only_x=c.groupby('ID').STATUS.apply(lambda s:s.eq('X').all())
print('Clientes somente X no histórico:',only_x.sum())
print('Somente X na base final:',d.ID.isin(only_x[only_x].index).sum())
assert d.ID.is_unique
assert (d.TARGET_90<=d.TARGET).all()
assert not set(FEATURES)&{'ID','PERFIL_ID','TARGET','TARGET_30','TARGET_90','MESES_HISTORICO','MES_RECENTE','MES_ANTIGO'}

Clientes somente X no histórico: 4536
Somente X na base final: 3347


## 3. Escala e features
Idade e emprego são convertidos de dias para anos sem arredondamento. Renda usa log1p e renda por membro, sem supor periodicidade. Não há winsorização nem corte arbitrário do tamanho familiar. StandardScaler e OneHotEncoder ficam no Pipeline, ajustados após a separação. FLAG_MOBIL é excluída por ser constante na base; CODE_GENDER fica fora desta versão, mas outros atributos podem conter proxies, o que ainda requer avaliação antes de uso real.

In [4]:
d.to_csv(OUT/'dataset_tratado.csv',index=False)
pd.DataFrame({'feature':FEATURES}).to_csv(OUT/'features.csv',index=False)
print('Base salva:',d.shape)
display(d[FEATURES+['TARGET']].head())

Base salva: (36457, 30)


,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,IDADE_ANOS,EMPREGO_SENTINELA,ANOS_EMPREGADO,LOG_RENDA,RENDA_POR_MEMBRO,TARGET
0,Y,Y,0,Working,Higher education,Civil marriage,Rented apartment,1,0,0,NaN,2.0,32.867899,0,12.435318,12.965712,213750.0,1
1,Y,Y,0,Working,Higher education,Civil marriage,Rented apartment,1,0,0,NaN,2.0,32.867899,0,12.435318,12.965712,213750.0,1
2,Y,Y,0,Working,Secondary / secondary special,Married,House / apartment,0,0,0,Security staff,2.0,58.792608,0,3.104723,11.630717,56250.0,0
3,N,Y,0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,0,1,1,Sales staff,1.0,52.320329,0,8.353183,12.506181,270000.0,0
4,N,Y,0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,0,1,1,Sales staff,1.0,52.320329,0,8.353183,12.506181,270000.0,0


## 4. Contrato para modelagem
Somente FEATURES entra em X. IDs, perfil, duração, datas, status e todos os alvos auxiliares permanecem para auditoria. Um target único será usado em treinamento, seleção e avaliação. A separação acontece no notebook 03.